In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text


server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

def nombre_mes_anio(fecha_mes_base):
    from datetime import datetime

    fecha = datetime.strptime(fecha_mes_base, "%Y-%m-%d")

    meses = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]

    return f"{meses[fecha.month - 1]} {fecha.year}"


In [2]:

filename='ACUM_DESEM.txt'

ruta_archivo = os.path.join(ruta_alfin, filename)
df_acumulado_cli = pd.read_csv(ruta_archivo,sep='|')

df_acumulado_cli["FECHA_DESEMBOLSO"] = pd.to_datetime(df_acumulado_cli["FECHA_DESEMBOLSO"], errors="coerce")

fecha_min = df_acumulado_cli["FECHA_DESEMBOLSO"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_acumulado_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.Alfin_ventas_desembolso
        WHERE FECHA_DESEMBOLSO >= '{fecha_desembolso}'
          AND FECHA_DESEMBOLSO <= EOMONTH('{fecha_desembolso}');
    """))

df_acumulado_cli.to_sql(
    name="Alfin_ventas_desembolso",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)




2026-07-01 00:00:00


496

In [3]:
filename='TARGET.txt'

ruta_archivo = os.path.join(ruta_alfin, filename)
df_desembolso_varios_cli = pd.read_csv(ruta_archivo,sep='|')

df_desembolso_varios_cli["FECHA_DESEMBOLSOS"] = pd.to_datetime(df_desembolso_varios_cli["FECHA_DESEMBOLSOS"], errors="coerce")

fecha_min = df_desembolso_varios_cli["FECHA_DESEMBOLSOS"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_desembolso_varios_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)

2026-07-02 00:00:00


In [5]:
fecha_desembolso

'2026-07-01'

In [4]:


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.Alfin_ventas_desembolso_varios
        WHERE FECHA_DESEMBOLSOS >= '{fecha_desembolso}'
          AND FECHA_DESEMBOLSOS <= EOMONTH('{fecha_desembolso}');
    """))


In [8]:
df_desembolso_varios_cli.head()

,DNI,CUENTA_BT,N_OPER,COD_SUCURSAL,SUCURSAL,MONTO_FINANCIADO,FECHA_SOL,FECHA_DESEMBOLSOS,TEA,CANAL,TIPO_DESEM,CODIGO_ID,CAMPAÑA
0,44570838,5018247,8465508,4270,SULLANA,2000.0,2026-07-17,2026-07-17,102.0,TARGET,DERIVACION,00000001,julio 2026
1,40846124,18198312,8465536,2249,VILLA EL SALVADOR 2,16300.0,2026-07-17,2026-07-17,53.0,TARGET,DERIVACION,00000001,julio 2026
2,40507683,18198213,8465409,4270,SULLANA,6000.0,2026-07-17,2026-07-17,78.0,TARGET,DERIVACION,00000001,julio 2026
3,32760225,18198239,8465400,8381,EMANCIPACION,3000.0,2026-07-17,2026-07-17,75.0,TARGET,DERIVACION,00000001,julio 2026
4,27386106,18198283,8465484,4281,CHICLAYO BALTA,9200.0,2026-07-17,2026-07-17,65.5,TARGET,DERIVACION,1,julio 2026


In [5]:


df_desembolso_varios_cli.to_sql(
    name="Alfin_ventas_desembolso_varios",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)


23